# D4-05 Optional Trails - simple numerical example

**Authors**: Romain Sacchi (PSI)

**Contact**: romain.sacchi@psi.ch

## Purpose

This notebook is a guided, end-to-end first experience with TRAILS. It covers:

1. Loading a small example data package.
2. Exploring the technosphere (`A`) and biosphere (`B`) matrices.
3. Temporal routing and graph visualization.
4. Running temporal and static LCA, then plotting results.
5. Linking results to the FaIR climate emulator (radiative forcing + temperature).
## Theory (short)

TRAILS (Temporal Routing And Aggregation of Impacts across Life-cycle Systems)
extends classic LCA by making time an explicit dimension. Exchanges can carry
**temporal distributions** (discrete, normal, lognormal, uniform, triangular),
which are expanded into year offsets during traversal. For each calendar year
that becomes active, the system is solved and biosphere flows are accumulated
at their respective years. This yields **time-resolved inventories** and **impact
scores** that answer questions like *when impacts occur* and *which upstream
suppliers dominate over time*.

**Environment requirements (from `pyproject.toml`):**
- Python `>=3.10, <3.13`
- Dependencies are loaded from `requirements.txt`
- Optional extras: `testing` (pytest), `docs` (sphinx)


In [ ]:
# Core imports
from pathlib import Path
from datapackage import Package

from trails import (
    Trails,
    get_lcia_method_names,
    plot_temporal_scores,
    plot_rf,
    plot_temp,
    clear_cache,
)

## 1. Load the example data package

We use the small data package shipped in `tutorials/DAY 4 - Edges and Trails/assets/trails/example_data_package`.


**Caching note**: the first time you load a data package, TRAILS builds
        and caches the technosphere and biosphere matrices (and their indices).
        Subsequent runs reuse the cache for faster startup. If you change the
        data package or want a clean rebuild, use `clear_cache()` to remove the
        cached matrices and force regeneration on the next load.


In [ ]:
# Point to the example data package
package_path = Path('assets/trails/example_data_package/datapackage.json')
package = Package(str(package_path))

# Initialize Trails
trails = Trails(package)

In [ ]:
# Optional: clear cache to force rebuilding matrices on next load
clear_cache()

## 2. Explore matrices and indices (A, B)


TRAILS exposes the technosphere matrix `A` and biosphere matrix `B` as 3D arrays:

- `A` indexed by `(year, activity, activity)`
- `B` indexed by `(year, activity, flow)`

You can inspect shapes and a few entries to get a feel for the data.

In [ ]:
# Inspect matrix shapes (year, activity, activity) and (year, activity, flow)
print('A shape:', trails.A.shape)
print('B shape:', trails.B.shape)

`A` stores production and technosphere-to-technosphere exchanges, while `B` stored technosphere-to-biosphere exchanges. 

These are `sparse.COO` matrices.

In [ ]:
trails.A.coords

Let's check the value of the production exchange of activity `0` at year `0`

In [ ]:
trails.A[
    0, # Year 0 or 2005
    0, # Activity 0
    0  # Product 0
]

Let's check the inputs of activity `13` at year 45 (2050).

In [ ]:
trails.A[
    45, # Year 0 or 2005
    13, # Activity 0
    :  # all
].data

In principle, we should be seeing the same here:

In [ ]:
# Print exchanges for a chosen activity index (example: 13)
# Replace 13 with an id from search_activity if desired.
trails.print_exchange_table(year=2050, act_idx=13)

You can also search activities by name and inspect exchanges for a
        specific activity.


In [ ]:
from trails import search_activity

# Find activity ids by keyword
search_activity(trails, name='electricity')[:5]

## 3. Choose an LCIA method

TRAILS exposes a helper that lists all available LCIA methods in the package.


In [ ]:
methods = get_lcia_method_names(trails)
methods[:10]

Pick a method and keep it in a single-element list (the LCA runner expects a list).

In [ ]:
methods = [
    "IPCC 2021 - climate change: total (excl. biogenic CO2) - global warming potential (GWP100)",
    "IPCC 2021 (incl. biogenic CO2) - climate change: total (incl. biogenic CO2) - global warming potential (GWP100)"
]

## 4. Run the temporal LCA

This computes time-resolved scores and inventory. We also keep the inventory so we
can later use the climate emulator.


### 4.1 Temporal routing (graph traversal)

This step constructs a temporal routing graph from a starting activity.
It is useful for understanding time-dependent dependencies before solving.

In [ ]:
# Example temporal routing (graph traversal)
trails.temporal_routing(
    start_year=2050,
    start_act_idx=13, # activity id
    amount=200000.0, # function unit amount
    max_depth=3, # supply chain levels to traverse up to
)

You can visualize the routing graph with `plot_temporal_graph`.
It is saved as an HTML in the same folder as this notebook.


In [ ]:
from trails.plotting import plot_temporal_graph

plot_temporal_graph(
    trails,
    filename='trails_graph.html',
)

### 4.2 Temporal LCA (full solve)

This computes time-resolved scores and inventory. We keep the inventory for the climate emulator later on. If you do not intend to use the emulator, you can probably disable inventory storage (faster). The higher `max_depth` in `temporal_routing()`, the more likely it is that the number of years to iterate increases, though.

In [ ]:
trails.lca(
    methods=methods,
    show_progress=True,
    compute_score=True,
    store_inventory=True,
)

We now have access to `trails.inventory` and `trails.characterized_inventory`.
For `trails.inventory`, we have amounts emitted:

* activity
* elementary flow
* year
* root activity, which is the first-level activity at teh root of all upstream emissionss (useful for plotting)

In [ ]:
trails.inventory.coords

For `trails.characterized_inventory`, it is similar, but it stores instead characterized results, with one additional dimension:
* method

In [ ]:
trails.characterized_inventory.coords

### 4.3 Static LCA (single year)

For convenience, this provides a quick single-year check for a chosen activity.


In [ ]:
# Example static LCA for a specific activity index
# Replace act_idx with an index of interest
trails.static_lca(year=2050, act_idx=13, methods=methods, amount=200000.0)
trails.static_score

## 5. Plot temporal impacts

The default plot highlights the top contributors over time.


In [ ]:
from trails.plotting import plot_temporal_scores
figs = plot_temporal_scores(
    stacked=False,
    legend_top_n=7,
    show_flow_contributions=False,
    trails=trails,
    title="",
    method_label="kg CO₂-eq",
    cumulative=False,
    width=550,
    height=450,
    year_tick=5,
    year_range=(2000, 2100),
    reference_year=2050,
    show_cumulative_axis=True,
    static_score=trails.static_score,
    static_score_dash= "dot",
    static_score_color= "red",
)

In [ ]:
figs[0]

In [ ]:
figs[1]

### Sankey

We can also produce a Sankey diagram (although this is not optimized yet).

In [ ]:
from trails.lca import build_temporal_sankey_tree
from trails.plotting import plot_temporal_sankey_graphlike

fig = plot_temporal_sankey_graphlike(
  trails,
  edge_weight="score",
  orientation="depth_x_year_y",
  #y_padding=0.000,
  filename="trails_sankey_graphlike.html",
  amount_label="kg CO₂‑eq",
   min_display_value=1e-12
)
fig

However, once emissions are distributed over time, static climate indicators such as **GWP100** are less appropriate because they collapse dynamic, time‑dependent effects into a single horizon. Moreover, background concentrations evolve over time, so fixed GWP factors cannot remain correct and a dynamic approach is needed.

## 6. Climate emulator ([FaIR](https://github.com/OMS-NetZero/FAIR))

TRAILS can translate inventory time series into **radiative forcing** and
**temperature anomaly** using the [FaIR](https://github.com/OMS-NetZero/FAIR) climate model.

Essentially, for each greenhouse gas (22 species, see mapping under `trails/data/scenarios/fair_species_map.yaml`), it does **two runs**:

* one run with background concentrations of a selected IAM scenario
* one run with background concentrations of a selected IAM scenario, **plus the (scaled) emissions from the LCA inventory**

If the additional emissions in the LCA inventory are too small, they are scaled to 1% of annual emissions to preserve the emulator's numerical stability. Since the impulse response model is largely linear, this should be without much impacts.

It then returns the delta between these two runs, for an ensemble of climate configurations, to reflect the contribution of the LCA system with respect to **radiative forcing** and **temperature anomaly**.

Outputs are stored as quantiles across all FaIR configurations:
`2.5, 25, 50, 75, 97.5`.

On some FaIR and SciPy combinations, concurrent per-species FaIR runs can fail
while building the stochastic climate ensemble. For a stable classroom workflow,
the call below keeps `per_species_workers=1`.

The following background concentration scenarios can be used:

#### Provided by FAiR
* high-overshoot
* high-extension
* verylow
* verylow-overshoot
* low
* medium-extension
* medium-overshoot

#### Extracted from IAM scenarios
* REMIND|SSP1-NPi2025
* REMIND|SSP1-PkBudg1000
* REMIND|SSP1-PkBudg650
* REMIND|SSP2-EU21-NDC
* REMIND|SSP2-EU21-NPi2025
* REMIND|SSP2-EU21-PkBudg1000
* REMIND|SSP2-EU21-PkBudg650
* REMIND|SSP2-NDC
* REMIND|SSP2-NPi2025
* REMIND|SSP2-PkBudg1000
* REMIND|SSP2-PkBudg650
* REMIND|SSP3-rollBack
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP1 - Low Emissions
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP1 - Very Low Emissions
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP2 - Low Emissions
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP2 - Low Overshoot
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP2 - Medium Emissions
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP2 - Medium-Low Emissions
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP2 - Very Low Emissions
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP3 - High Emissions
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP4 - Low Overshoot
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP5 - High Emissions
* MESSAGEix-GLOBIOM-GAINS 2.1-M-R12|SSP5 - Low Overshoot
* IMAGE 3.4|SSP1_L
* IMAGE 3.4|SSP1_M
* IMAGE 3.4|SSP1_VLLO
* IMAGE 3.4|SSP2_L
* IMAGE 3.4|SSP2_M_CP
* IMAGE 3.4|SSP2_VLHO
* IMAGE 3.4|SSP3_H
* IMAGE 3.4|SSP5_H 

This allows the use of a background-emission scenario consistent with the LCI background scenarios to provide consistent radiative forcing and temperature anomalies. Note that the initial emissions time series provided by `FAiR` start in the year 1750 and ends in 2500. For the IAM scenarios, we only have emissions data from 2005 to 2100; we "freeze" the 2100 emission levels until 2500. It is unclear how much of an impact this has on the resulting radiative forcing and temperature anomaly.

In [ ]:
run_fair_delta_rf?

In [ ]:
from trails.fair_rf import run_fair_delta_rf

trails.debug = False
delta = run_fair_delta_rf(
    trails,
    scenario="REMIND|SSP3-rollBack",
)

In [ ]:
# Access stored results
trails.instant_radiative_forcing

In [ ]:
trails.delta_temperature

### Plot radiative forcing (median + quantile band)

By default, this plots the 50th quantile values from the ensemble of climate configurations.

In [ ]:
fig_rf = plot_rf(
    trails,
    by='root activity', # or by "flow"
    year_range=(2000, 2200),
    year_tick=10,
    reference_year=2050,
)
fig_rf

### Plot temperature anomaly (median + quantile band)

`plot_temp()` visualizes the time‑resolved temperature anomaly attributable to the modeled system, computed by allocating FaIR’s temperature response to the system’s emissions (by flow or by root activity) over time, with the median (50th percentile) across FaIR configurations by default and an optional 2.5–97.5% uncertainty band.

In [ ]:
fig_temp = plot_temp(
    trails,
    by='root activity', # or by="flow"
    year_range=(2000, 2200),
    year_tick=10,
)
fig_temp

## 8. Next steps

- Import user inventories (Excel) with `trails.import_excel_inventory`: check the other notebook.
